# Лекция. Векторные вычисления

In [1]:
import numpy as np 
import pandas as pd

## § 1. Зачем нужны векторные вычисления

В двух словах, векторные вычисления — это скорость. 

### Пример

Допустим, у нас есть список из 100 миллионов чисел, и нам нужно каждое из них умножить на 2. 
Создадим массив, содержащий 100 миллионов случайных чисел из стандартного нормального распределения:

In [2]:
big_list_of_numbers = np.random.randn(100000000)
big_list_of_numbers

array([ 0.46049856,  0.98007767, -1.68840935, ..., -1.4670008 ,
        0.39280483,  0.22062263])

Подключим модуль для замера времени:

In [3]:
import time

И применим старый добрый цикл `for`:

In [4]:
start_time_for_loop = time.time()  # Засекаем старт

result_for_loop = [] 
for number in big_list_of_numbers:
    result_for_loop.append(2*number)

time_for_loop = time.time() - start_time_for_loop  # Стоп! Вычисляем время

print('Время выполнения с циклом for:', round(time_for_loop, 4), 'секунд.')

Время выполнения с циклом for: 13.7707 секунд.


Теперь сделаем то же самое при помощи генератора списков:

In [ ]:
start_time_generator = time.time()  # Засекаем старт

result_generator = [2*x for x in big_list_of_numbers] 

time_generator = time.time() - start_time_generator  # Стоп! Вычисляем время

print('Время выполнения с генератором:', round(time_generator, 4), 'секунд.')

Разница, честно говоря, невелика. А теперь сделаем то же самое при помощи векторой операции:

In [ ]:
start_time_vector = time.time()  # Засекаем старт заново

result_vector = big_list_of_numbers * 2  

time_vector = time.time() - start_time_vector  # Стоп! Вычисляем время

print('Время выполнения векторной операции:', round(time_vector, 4), 'секунд.')

И сравним скорость:

In [ ]:
print('Цикл:',round(time_for_loop, 4), 'секунд.')
print('Генератор:',round(time_generator, 4), 'секунд.')
print('Векторная операция:',  round(time_vector, 4), 'секунд.')
print('')
print('Ускорение по сравнению с циклом в', round(time_for_loop / time_vector), 'раз.')
print('Ускорение по сравнению с генератором в', round(time_generator / time_vector), 'раз.')

### Но как?

И все-таки как работают векторные вычисления с точки зрения алгоритмики? Ведь процессор не может одновременно выполнять 100 миллионов операций. Значит, он все равно выполняет их последовательно, то есть — исполняет цикл. И это действительно так. 

Но весь секрет — в том, как он выполняется и кто его выполняет.

### Цикл

В цикле `for` вычисления происходят на языке Python, а Python — это интерпретируемый язык. Его код выполняется построчно, что, безусловно, делает разработку очень гибкой, но сильно (точнее — очень сильно) замедляет работу. Для каждой из 100 миллионов итераций интерпретатор должен:
1) проверить тип объекта (это действительно float?),
1) найти оператор умножения для этого типа,
1) выделить память для нового объекта-результата,
1) добавить ссылку на этот объект в список result.

Это огромные накладные расходы на каждую элементарную операцию. Можно сказать, что 99% времени тратится не на умножение, а на «административную работу» интерпретатора.

### Векторная операция

При выполнении векторной операции умножения интерпретатор Python видит одну операцию (умножение) между двумя объектами (массивом и числом). Он передает управление скомпилированной библиотеке numpy, написанной на языках C/C++/Fortran. Та в свою очередь, получив указатели на начало массива в памяти и его размер, запускает цикл на C/C++/Fortran, а C и Fortran — это классические компилируемые языки. 

Перед запуском исходный код преобразуется компилятором непосредственно в машинные инструкции (бинарный файл), что обеспечивает максимальную производительность при расчётах. В этом цикле:

1) типы данных известны заранее (float64), поэтому не нужно проверять типы на каждой итерации,
2) не нужно создавать новые объекты для каждого результата,
3) не нужно выделять память под каждый полученный результат, так как память выделяется одним большим блоком сразу на весь массив.

Итог — выполняется та же последовательность действий, но без накладных расходов интерпретатора. Поэтому векторная операция выполняется в десятки (сотни) раз быстрее.

### Что под капотом у pandas?

Все дело в том, что под капотом у pandas есть основа — это массивы numpy, а сами датафреймы pandas —  это надстройка над массивами.

**numpy**: предоставляет «голый» высокоскоростной массив (np.array) и фундаментальные математические операции для него (сложение, умножение, сравнение и т.д.). Это движок для вычислений.

**pandas**: берет мощный, но простой массив numpy, оборачивает его в удобную структуру с метками (датафрейм с индексами для строк и именами для столбцов) и предоставляет богатый API для работы с табличными данными (загрузка, объединение, группировка).

Продемонстрируем эту связь на примере несложного датафрейма. Сам по себе он представляет собой структуру pd.DataFrame:

In [ ]:
data = {'Возраст': [25, 30, 35], 'Зарплата': [50000, 60000, 75000]}
df = pd.DataFrame(data)
display(df)
type(df)

Заглянем под капот. Столбец датафрейма — это объект, относящийся к структуре pd.Series:

In [ ]:
age_series = df['Возраст']
display(age_series)
type(age_series)

А что внутри Series? Метод `values` возвращает массив numpy:

In [ ]:
age_numpy_array = age_series.values
display(age_numpy_array)
type(age_numpy_array)

В этом примере, кроме всего прочего, мы обнвружили, что между основой (np.array) и надстройкой (pd.DataFrame) есть еще и прослойка (pd.Series). Серии сами по себе тоже весьма интересны, мы поговорим о них чуть позже.

Когда мы работаем с данными в Pandas и хотим делать быстрые математические преобразования, мы «спускаемся» на уровень массивов numpy (часто даже не замечая этого), используя методы pandas, которые внутри сами работают как векторные операции.
При этом numpy — это основа, так как он обеспечивает массивы и базовые векторные операции, а pandas — это удобная обертка, так как он хранит данные в виде таблиц (pd.DataFrame), состоящих из столбцов (pd.Series), которые внутри являются массивами (np.array).

Сегодняшняя лекция — о том, как мыслить «векторно» и применять этот подход с первых шагов в Data Science. 

## § 2. Массивы np.array

Наша ближайшая цель — научиться создавать массивы numpy, и разобраться с их атрибутами. 

Массив в numpy представляет собой многомерную таблицу элементов **одного типа** (этим он отличается от списков, элементы которых могут относиться к разным типам), расположенных в памяти непрерывным блоком. Их можно создавать разными способами.

Рассмотрим основные из них.

### Из списка

Берем список целых чисел и применяем метод `np.array`:

In [ ]:
list_1d = [1, 2, 3, 4, 5]
array_1d = np.array(list_1d)
array_1d

Пусть теперь какое-нибудь одно число относится к типу чисел с плавающей запятой:

In [ ]:
list_1d = [1, 2, 3, 4.5, 5]
array_1d = np.array(list_1d)
array_1d

В результате все числа массива стали дробными (если после десятичного разделителя ничего нет, то это значит, что дробная часть числа равна нулю, но число все равно считается дробным).

А теперь пусть какое-нибудь одно число записано как строка:

In [ ]:
list_1d = [1, 2, 3, 4, '5']
array_1d = np.array(list_1d)
array_1d

Обратите внимание на то, что все элементы массива стали строками. 

В этом выводе `dtype='<U11'` означает тип данных в библиотеке numpy, который обозначает строковый тип с фиксированной длиной 11 символов в кодировке Unicode. Расшифровка обозначения:
- U — обозначает тип данных Unicode,
- 11 — указывает на количество символов, которое может содержать строка. 

Каждый массив numpy имеет ассоциированный `dtype`, который определяет тип элементов, хранящихся в массиве. 
Важно, что в массивах numpy все элементы должны иметь единый тип данных. Если в массиве присутствуют элементы разных типов, numpy может автоматически определить общий тип на основе преобладающих данных.

### Из вложенного списка

Берем список списков одинаковой длины и снова применяем метод `np.array`:

In [ ]:
list_2d = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
array_2d = np.array(list_2d)
array_2d

Получаем матрицу 3 на 3. Причем, такой вывод (в отличие от вывода вложенного списка) действительно указывает на матричную природу этого объекта.

>#### Задание
Проверьте, что будет, если один из элементов окажется строкой? А если длины вложенных списков будут неравны?

In [ ]:
# Код

### Массивы нулей или единиц

Массивы нулей создаются при помощи метода `np.zeros`:

In [ ]:
zeros_array_1d = np.zeros(5)
zeros_array_1d

Все элементы оказались дробными. Это можно исправить, если явно указать атрибут `dtype`:

In [ ]:
zeros_array_1d = np.zeros(5, dtype = int)
zeros_array_1d

Двумерные массивы нулей получаются, если передать методу `np.zeros` в качестве аргумента кортеж из двух чисел (первое указывает на количество строк, а второе — на количество столбцов в результирующем матрице):

In [ ]:
zeros_array_2d = np.zeros((3, 4))
zeros_array_2d

Аналогично, при помощи метода `np.ones` создаются массивы, заполненные единицами.

>#### Задание
>1) Создайте одномерный массив длины 10, заполненный единицами. 
>2) Добейтесь, чтобы единицы относились к типу целых чисел.
>3) А теперь — чтобы они были строками.
>4) Создайте двумерный массив, заполненный единицами так, чтобы у результирующей матрицы было 2 строки и 5 столбцов.

In [ ]:
zeros_array_1d = np.ones((2, 5), dtype = int)
zeros_array_1d

### Равномерные массивы

В numpy есть аналог встроенной функции `range`, он вызывается при помощи метода `np.arange`:

In [ ]:
np.arange(10)

Отличие в том, что он возвращает массив, а не объект типа `range`. Общий синтаксис тоже аналогичен:
```python
np.arange(start, stop, step)
```
Кроме того, можно выстраивать равномерные массивы с заданным числом точек. Для этого используется метод `np.linspace`. В отличие от `np.arange`, который задаёт шаг между элементами, `np.linspace` указывет точное количество элементов, которые нужно получить в заданном интервале (шаг при этом вычисляется). Общий синтаксис таков:
```python
np.linspace(start, stop, N)
```

при этом, в отличие от `range` и `np.arange`, правая граница включается. Например:

In [ ]:
np.linspace(5, 10, 10)

### Контроль над скоростью и точностью

Точно контролировать, как хранятся данные, позволяет атрибут `dtype`, определяющий тип данных, который, в свою очередь, напрямую влияет на скорость вычислений и потребление памяти. Нпаример, можно уменьшить потребление памяти:

- **float64** — стандартный тип по умолчанию, он задает высокую точность,

- **float32** — занимает в 2 раза меньше памяти, но обладает меньшей точностью (в связи с чем, часто используется в deep learning, когда важна именно скорость, а не предельная точность).

Инструментом для конвертации служит метод `astype`. Например, можно перевести массив в целый тип:

In [ ]:
np.linspace(5, 10, 5)

In [ ]:
np.linspace(5, 10, 5).astype(int)

Правда, при этом мы сильно потеряли в точности (обратите внимание, что происходит даже не округление, а грубое отбрасывание дробной части), но зато выиграли в скорости. Хотя, не факт, что оно того стоило.

### Булевы маски

В обычном Python, чтобы оставить в списке только элементы, удовлетворяющие какому-то условию, мы бы писали цикл с условием `if`. В numpy мы используем булевы маски. Маска — это массив того же размера, что и исходный, состоящий только из True и False. Если мы «наложим» такую маску на массив при помощи синтаксической конструкции
```python
array[mask]
```
то numpy оставит только те элементы, где в маске находится True. Вспомним, что у нас есть большой массив из 100 миллионов чисел:

In [ ]:
big_list_of_numbers

Отфильтруем этот массив, оставив в нем только положительные числа. Сначала при помощи цикла:

In [ ]:
start_time_for_loops = time.time()

positive = []

for number in big_list_of_numbers:
    if number > 0:
        positive.append(number)

time_for_loops = time.time() - start_time_for_loops

positive_numbers = np.array(positive)
display(positive_numbers)
print('Время исполнения цикла:', time_for_loop, 'секунд.')

Затем при помощи генератора списков:

In [ ]:
start_time_generator = time.time()

positive = [x for x in big_list_of_numbers if x > 0]

time_generator = time.time() - start_time_generator

positive_numbers = np.array(positive)
display(positive_numbers)
print('Время исполнения генератора:', time_generator, 'секунд.')

А теперь — при помощи маски:

In [ ]:
start_time_mask = time.time()

mask = big_list_of_numbers > 0
positive_numbers = big_list_of_numbers[mask]

time_mask = time.time() - start_time_mask
display(positive_numbers)
print('Время исполнения булевой маски:', time_mask, 'секунд.')

In [ ]:
print('Цикл:',round(time_for_loop, 4), 'секунд.')
print('Генератор:',round(time_generator, 4), 'секунд.')
print('Векторная операция:',  round(time_vector, 4), 'секунд.')
print('')
print('Ускорение по сравнению с циклом в', round(time_for_loop / time_vector), 'раз.')
print('Ускорение по сравнению с генератором в', round(time_generator / time_vector), 'раз.')

Понятно, что маска работает быстрее. 

Но это еще не все. Следует понимать, что сравнение, использованное нами в цикле: 
```python
number > 0
```
это условие, которое может быть истинным или ложным. А сравнение, использованное в маске:
```python
big_list_of_numbers > 0
```
это команда numpy: «сравни каждый элемент с нулем и верни мне массив ответов».

In [ ]:
big_list_of_numbers > 0

Это исключительная фишка массивов numpy. Если попытаться применить ее в обычному списку, то ничего не получится:

In [ ]:
# [1, 2, 3] > 0

Маски могут содержать сложные составные условия:

In [ ]:
mask = (big_list_of_numbers > -big_list_of_numbers.std()) & (big_list_of_numbers < big_list_of_numbers.std())
condition_numbers = big_list_of_numbers[mask]
condition_numbers

Так мы получили только те числа, которые отстоят от нуля (как налево, так и направо) меньше, чем на сигму.

>#### Задание
Известно правило трех сигм: если случайная величина распределена нормально, то отклонение ее значений от математического ожидания на величину, большую чем три сигма, практически невозможно. Получите из массива big_list_of_numbers эти практически невозможные значения. Сколько их получилось?

In [ ]:
# Код

Итак, маски — мощный инструмент для построения сложных, реальных условий фильтрации. В бизнес-анализе такие запросы («найти всех, кто купил товар X и при этом был из региона Y») — это и есть хлеб насущный, и они решаются именно через векторные логические операции.

## § 3. Серии pd.Series

Если `np.array` — это просто ряд чисел, то `pd.Series` — это тот же массив, но у каждого элемента есть «имя» (Index). Данные лежат внутри  серии в виде знакомого нам массива `np.array`, а индексы позволяют обращаться к данным не только по номеру (как это происходит в списках и массивах), но и по смыслу (например, по дате или имени студента).

Объекты `pd.Series` могут создаваться из списков, словарей и массивов numpy. 

### Из списка

Берем список строк (хотя, с таким же успехом могли бы взять список чисел или булевых значений) и применяем метод `pd.Series`:

In [ ]:
names_list = ['Петя', 'Катя', 'Женя']
s = pd.Series(names_list)
s

Получилось что-то очень похожее на список, но с явным указанием индексов. Обратите внимание, что индекс — это не список и не массив, это специальный тип индекса, характериный для библиотеки pandas:

In [ ]:
s.index

По умолчанию используется именно он. Но можно задать собственный индекс. Например:

In [ ]:
names_list = ['Петя', 'Катя', 'Женя']
s = pd.Series(names_list, index = ['a', 'b', 'c'])
s

Теперь у имен появились строковые метки, но и они по-прежнему не являются списком. Это по-прежнему специальный тип индекса (только другого):

In [ ]:
s.index

В уже существующей серии индекс (при желании) можно поменять:

In [ ]:
s.index = ['A', 'B', 'C']
s

Обратите внимание, что, несмотря на то, что в качестве нового индекса мы указали список, после того, как он становится индексом, он перестает быть списком:

In [ ]:
s.index

Значения серии вызываются методом `values`:

In [ ]:
s.values

И они тоже не являются списком. Но это не специализированный объект библиотеки pandas. Это уже хорошо знакомый нам массив `np.array` со всеми вытекающими из этого последствиями: например, понятно, что серии — это надстройки над массивами (массивы с индексами).

### Из словаря

Логика создания серии из словаря вполне очевидна: давайте передадим словарь в серию так, чтобы ключи были элементами индекса, а значения — собственно значениями серии:

In [ ]:
names_dict = {'a': 'Петя', 'b': 'Катя', 'c': 'Женя'}
s = pd.Series(names_dict)
s

### Из массива numpy

По большому счету, это ничем не отличается от создания серии из списка. Просто вместо списка значений используется массив:

In [ ]:
names_array = np.array(names_list)
s = pd.Series(names_array)
s

### Векторные операции в Series

Хорошая новость: всё, что мы говорили про np.array, работает и в pd.Series. Сложение, умножение, булевы маски — у всего этого синтаксис остается тем же. Pandas просто «пробрасывает» операцию во внутренний массив numpy.

Например, добавим ко всем именам статус 'студент':

In [ ]:
s_student = 'студент ' + s
s_student

При этом Катя стала студентом, а не студенткой, но это не важно (тем более, что мы все равно не знаем, какого пола Женя). Важно то, что это была векторная операция, и, если бы у нас было не три студента, а три миллиона студентов, то мы смогли бы присвоить им статус, не прибегая к использованию цикла.

А еще — важно то, что мы, так же, как и в случае с массивами, можем устраивать фильтрацию по булевой маске. Например, давайте создадим вспомогательный массив с оценками наших студентов и передадим его в серию `score`, использовав в качестве индекса значения серии `s`:

In [ ]:
score = pd.Series(np.array([5, 5, 3]), index = s.values)
score

Петя и Катя — отличники, а Женя — троечник (или троечница, без разницы). В качестве условия для фильтрации будем использовать тот факт, что студент должен быть отличником:

In [ ]:
mask = score == 5
mask

Но такая маска никуда не годится, потому что получилось так, что `mask` — это снова серия, а должен быть массив numpy. Поэтому в качестве маски будем использовать не `mask`, а `mask.values` (это как раз массив):

In [ ]:
s_A_student = s[mask.values]
s_A_student

И мы отфильтровали отличников.

### Выравнивание данных

В numpy массивы просто складываются по позициям. В pandas происходит выравнивание по индексам. 

А именно: при сложении двух серий pandas ищет одинаковые индексы, и если индекса нет хотя бы в одном из наборов — мы получаем NaN (Not a Number — пустота). То же самое происходит при умножении или делении. Это критически важно понимать, чтобы не терять данные, потому что столбцы датафреймов — это не что иное как серии, и если для получения новых признаков мы  начнем их складывать (умножать, делить), то при этом число пропусков в нашем датафрейме будет только расти.

Для иллюстрации того, как работает выравнивание, рассмотрим такой пример. Наши студенты — все те же Петя, Катя и Женя —написали две контрольные. Первую контрольную Петя и Катя написали на 5, а Женя — на 3. Вторую контрольную Петя написал на 5, Катя — на 4, а Женя — вообще не писал. 

Оформим эти результаты в виде двух серий:

In [ ]:
control_1 = pd.Series([5, 3, 5], index=['Катя', 'Женя', 'Петя'])
control_2 = pd.Series([5, 4], index=['Петя', 'Катя'])

display(control_1)
display(control_2)

Обратите вннимание, что индексы в этих сериях следуют в разных порядках (допустим, в тех порядках, в которых студенты сдавали свои работы в конце пары) — в контексте сложения двух серий это не имеет никакого значения, так как операция все равно будет выполняться по индексам, а не по позициям.

Дальше мы хотим получить средние баллы наших студентов. Для этого, не прибегая, разумеется к циклам (мы же умные!), мы оперируем непосредственно с сериями (как и положено по принципам векторных вычислений):

In [ ]:
average_score = (control_1 + control_2)/2
average_score

И вот что мы видим: у Жени нет оценки за вторую контрольную, и, как следствие, нет среднего балла. А это неправильно, потому что средний балл у него все-таки есть: 3/2 = 1.5 (хотя и не очень высокий).

Теперь давайте предположим, что студенты писали не две, а десять контрольных. Тогда, скорее всего, пропуски будут не только у Жени, но и у других студентов тоже (хотя бы по одному разу), и в результате подсчёта средних баллов мы получим сплошные NaN’ы, что будет, мягко говоря, малоинформативно.

>#### Задание
Придумайте, как исправить эту ситуацию, и реализуйте своё исправление в коде.

In [ ]:
control_2['Женя'] = 0
average_score = (control_1 + control_2)/2
average_score

## § 4. Интеграция в Data Science

Рассмотрим сведения об успеваемости студентов:

In [ ]:
a = pd.read_csv('data/student_scores.csv')
display(a.head(3))
a.info()

### Описание данных

Всего на потоке учатся 139 студентов в шести группах (группы нумеруются от A до F). Они пишут проверочные работы по математике, физике и английскому, но иногда прогуливают. Сведения представлены в таблице, структура сведений вполне очевидна:
- ID — ID студента: первая буква означает группу, следующие две цифры означают его номер в списке группы по алфавиту (тип данных — строка);
- math_1, math_2, math_3, physics_1, physics_2, english — оценки, полученные на контрольных работах (тип данных — целый).

Оценки выставляются по стобалльной шкале, они должны быть целыми числами, но из-за пропусков данных при чтении csv они стали дробными.

### Постановка задачи

Мы хотим поощрять студентов, успеваемость которых выше средней по потоку. Принцип поощрения такой:
1. формируем итоговую успеваемость каждого студента (это среднее всех его оценок по всем предметам),
2. если средняя оценка студента по математике выше средней оценки по математике всего потока, мы увеличиваем его итоговую оценку на 10%,
3. если средняя оценка студента по физике выше средней оценки по физике всего потока, мы увеличиваем его итоговую оценку на 5%,
4. если средняя оценка студента по английскому  выше средней оценки по английскому всего потока, мы увеличиваем его итоговую оценку на 2%.

### Заполнение пропусков
На всякий случай формируем копию датафрейма (чтобы нечаянно не исказить исходник) и заполняем пропуски нулями. Логика понятна: если студент пропустил контрольную, то он набрал на ней 0 баллов.

In [ ]:
b = a.copy()
b = b.fillna(0)
b


>#### Задание
Как по-другому можно было бы заполнить пропуски? 

### Преобразование типов

К целому типу относятся все столбцы кроме первого. Поэтому берем срез по списку столбцов и по этому срезу выполняем преобразование при помощи метода `astype`: 

In [ ]:
b[b.columns[1:]] = b[b.columns[1:]].astype(int)
b

### Формирование общей успеваемости
Фактически, чтобы получить общую успеваемость каждого студента, нам нужно вычислить среднее значение в каждой строке по всем столбцам, кроме первого.

#### Неправильное рещение — вложенный цикл

Мы разбираем здесь неправильное решение только для того, чтобы подчеркнуть разницу между циклами Python и векторными вычислениями.

Никогда так не поступайте:

In [ ]:
c = b.copy() # Создаем копию датафрейма, чтобы не испортить его рабочую версию
col_list = c.columns[1:] # Формируем список столбцов для внутреннего цикла

average_list = [] # Заводим пустой список для результата

for i in range(len(b)): # Внешний цикл свеху вниз
    S = 0
    for col in col_list: # Внутренний цикл слева направо
        S = S + b[col][i]
    S_average = round(S/len(col_list))
    average_list.append(S_average)

c['average'] = average_list # Форимрование нового столбца с общей успеваемостью
c

#### Правильное решение — векторные операции
Давайте немного порассуждаем. Мы можем посчитать средние значения при помощи метода `mean`. У него под капотом зашит как раз оптимизированный код на компилируемом (а не интерпретируемом) уровне.

In [ ]:
b[b.columns[1:]].mean()

Но так получаются средние значения по столбцам, а нам нужны средние по строкам — казалось бы, все равно нужен какой-то цикл слева направо. Но мы же можем и схитрить: давайте транспонируем наш датафрейм:

In [ ]:
b[b.columns[1:]].T

И уже к транспонированному датафрейму применим метод `mean`:

In [ ]:
b[b.columns[1:]].T.mean()

После таких предварительных рассуждений весь код занимает одну строчку:

In [ ]:
b['average'] = b[b.columns[1:]].T.mean().astype(int)
b

Обратите внимание, что для получения целых чисел мы применили не округление `round`, а преобразование типов `astype(int)`, которое работает грубее: оно просто отбрасывает дробную часть. Поэтому есть определённая разница в результатах, например, студент F24 по первому методу имел общую оценку 48, а по второму — имеет 47. Но, во-первых, разница число косметическая, во-вторых, мы могли бы и округление применить (мы применили преобразование типов просто для демонстрации разнообразия решений). 

А в-третьих, самое главное — не в этом, а в том, что мы не прибегали к циклам. 

### Средняя успеваемость на потоке по математике

#### Неправильное рещение — вложенный цикл

Никогда так не поступайте:

In [ ]:
c = b.copy() # Создаем копию датафрейма, чтобы не испортить его рабочую версию
col_list = c.columns[1:4] # Формируем список столбцов, относящихся именно к математике (для внутреннего цикла)

S = 0 # Обнуляем сумму

for i in range(len(b)): # Внешний цикл свеху вниз
    for col in col_list: # Внутренний цикл слева направо
        S = S + b[col][i]

m = round(S/(len(col_list)*len(c))) # Вычисляем среднее и округляем
m

#### Правильное решение — векторные операции
Опять-таки, давайте немного порассуждаем. Мы можем посчитать средние значения при помощи метода `mean`:

In [ ]:
b[b.columns[1:4]].mean()

Но ведь то, что получилось — это серия. И от нее тоже можно взять среднее. Это и будет результат:

In [ ]:
m = b[b.columns[1:4]].mean().mean()
m

Остается для красоты окрулить его до целого:

In [ ]:
m = round(b[b.columns[1:4]].mean().mean())
m

Снова решение заняло одну строчку, но самое главное — что мы не использовали циклы, а применили методы векторных вычислений. 

### Поощрение за математику

Нам понадобятся средние оценки по математике для всех студентов. Мы не станем сторить циклы, мы поступим интеллигентно:

In [ ]:
average_math = b[b.columns[1:4]].T.mean()
average_math

Средние оценки по математике готовы. 

А дальше мы порассуждаем сначала грубо (в категориях циклов), а затем изящно (в категориях векторных операций).

#### Неправильное рещение — цикл

Никогда так не поступайте:

In [ ]:
c = b.copy() # Создаем копию датафрейма, чтобы не испортить его рабочую версию

result = [] # Заводим пустой список для результатов

for i in range(len(c)): # Цикл свеху вниз
    if average_math[i] > m: # Прверяем условие
        result.append(round(c['average'][i]*1.1)) # Поощряем
    else:
        result.append(round(c['average'][i])) # Оставляем как есть
c['average'] = result # Преобразуем столбец к новым оценкам
c

#### Правильное решение — булева маска

Синтаксис масок на массивах вполне очевиден. Общая логика такова:

```python
array[mask]
```
— возврщает маскированный фрагмент массива, то есть, только те позиции, которые соответствуют позициям True в маске. Если же нужно преобразовать массив, то применяют:

```python
array[mask] = f(array[mask])
```
— где f — это какое-то преобразование (все равно, какое). При этом маскированные элементы будут преобразованы, а все остальные останутся без изменений. 

Похожая логика относится и к сериям:
```python
series[mask]
```
— возврщает маскированную серию, то есть, на позициях True будут элементы серии, а на позициях False будут NaN'ы. Если же нужно преобразовать серию, то применяют:

```python
series[mask] = f(series[mask])
```
— где f — это преобразование. При этом маскированные элементы будут преобразованы, а все остальные останутся без изменений.

Однако, все это верно только в том случае, если серия создана как самостоятельный объект. Если же она получается как столбец датафрейма, то нужно предварительно отделить ее от датафрема, применив метод `copy`, так как при работе с датафреймами используется другая логика (это логика локализаций, мы перейдем к ней позже, на следующих лекциях).

Сейчас мы очень подробно, по шагам разберем весь процесс поощрения студентов при помощи булевой маски. Итак, формируем маску. Получается серия pandas.

In [ ]:
mask = average_math > m
mask

Поскольку мы делали маску из серии, сама маска тоже оказалась серией. Но маска всегда должна быть массивом (по-другому не работает). Поэтому мы переводим ее в массив при помощи метода `values`:

In [ ]:
mask = mask.values
mask

Берем итоговые оценки из датафрейма и выводим их в самосотоятельную серию при помощи метода `copy`:

In [ ]:
average_new = b['average'].copy()
average_new

При меняем к самостоятельной серии преобразование по маске:

In [ ]:
average_new[mask] = average_new[mask]*1.1
average_new

И записываем новые оценки поверх старых:

In [ ]:
b['average'] = average_new.astype(int)
b

Нам опять удалось издежать построения циклов — в этом и состоит преимущество векторных операций (в том, что цикл исполняется под капотом: не на Python'е, а на других, очень быстрых языках).

>#### Задание
Ипользуя булевы маски (но ни в коем случае не циклы!), поощрите студентов за физику и за английский. Заметьте, что умножаться на мультипликатор должна именно обновлённая общая оценка,то есть — при этом будут возникать сложные проценты (проценты от процентов). Сравните обновленные результаты с теми, что были у них до поощрения:
>1. Каково теперь максимальное значение общей оценки? А какое было?
>2. У кого из студентов оказался наибольший мультипликаторный коэффициент?
>3. Какая доля студентов осталась вообще без поощрения?

# Домашнее задание

Снова загрузите данные об успеваемости. Теперь мы займемся поиском «ботаников» («ботаником» мы будем называть такого студента, у которого средняя оценка по каждому предмету превышает медианную оценку всего потока по этому предмету).

>#### Задание
>1. С помощью сложной булевой маски найдите таких студентов. Много ли их оказалось? И есть ли такие вообще?
>2. Если они есть, то каждому такому студенту увеличьте его общую оценку на 15%.

#### Замечание
Выше мы искали среднее по датафрейму как среднее от среднего столбцов, и это действительно так, потому что среднее значение относится к классу линейных статистик. Но имейте в виду, что общая медиана двумерной таблицы так не вычисляется. Она не равна медиане от медиан столбцов, так как медиана — это одна из нелинейных статистик. 

Для вычисления можно (опять-таки, избегая построения цикла!) применить векторную операцию `flatten`, которая «разворачивает» матрицу в вектор. Синтаксис поиска общей медианы с ее помощью таков:
```python
np.median(df.values.flatten())
```
где df — это, как обычно, какой-нибудь датафрейм (или его фрагмент).